# Deep Past Initiative - Akkadian → English (Inference Only)

Load fine-tuned flan-t5-base from Kaggle Dataset and generate predictions. **No internet required.**

In [ ]:
import os
import re

import pandas as pd
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

# ── Paths ──
# Find competition data
DATA_DIR = "/kaggle/input/deep-past-initiative-machine-translation"
if not os.path.exists(f"{DATA_DIR}/test.csv"):
    for root, dirs, files in os.walk("/kaggle/input"):
        if "test.csv" in files and "train.csv" in files:
            DATA_DIR = root
            break

# Find fine-tuned model — adjust dataset name if different
MODEL_DIR = None
for candidate in [
    "/kaggle/input/akkadian-flan-t5-finetuned/best_model",
    "/kaggle/input/akkadian-flan-t5-finetuned",
]:
    if os.path.exists(candidate) and os.path.exists(os.path.join(candidate, "config.json")):
        MODEL_DIR = candidate
        break

# Fallback: search all input dirs for config.json
if MODEL_DIR is None:
    for root, dirs, files in os.walk("/kaggle/input"):
        if "config.json" in files and "model.safetensors" in files:
            MODEL_DIR = root
            break

PREFIX = "translate Akkadian to English: "
MAX_SOURCE_LEN = 256
MAX_TARGET_LEN = 256

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
print(f"Data dir: {DATA_DIR}")
print(f"Model dir: {MODEL_DIR}")

In [ ]:
def clean_transliteration(text: str) -> str:
    """Clean Akkadian transliteration following competition guidelines."""
    if not text or not isinstance(text, str):
        return ""
    t = text
    t = t.replace("Ḫ", "H").replace("ḫ", "h")
    t = t.replace("˹", "").replace("˺", "")

    GAP, BIGGAP = "___GAP___", "___BIGGAP___"
    t = re.sub(r"\[x\]", GAP, t)
    t = re.sub(r"\[\s*…\s*…?\s*\]", BIGGAP, t)
    t = re.sub(r"\[\s*\.\.\.\s*\.?\.?\.?\s*\]", BIGGAP, t)
    t = re.sub(r"\[([^\]]*)\]", r"\1", t)
    t = re.sub(r"<<[^>]*>>", "", t)
    t = re.sub(r"<([^>]*)>", r"\1", t)
    t = re.sub(r"…", f" {BIGGAP} ", t)
    t = t.replace(GAP, " <gap> ").replace(BIGGAP, " <big_gap> ")

    t = re.sub(r"!", "", t)
    t = re.sub(r"\?", "", t)
    t = re.sub(r"/", " ", t)
    t = re.sub(r"\s*:\s*", " ", t)
    t = t.translate(str.maketrans("₀₁₂₃₄₅₆₇₈₉ₓ", "0123456789x"))
    t = t.translate(str.maketrans("⁰¹²³⁴⁵⁶⁷⁸⁹", "0123456789"))
    t = re.sub(r"(<big_gap>\s*)+", "<big_gap> ", t)
    t = re.sub(r"(<gap>\s*)+", "<gap> ", t)
    return re.sub(r"\s+", " ", t).strip()

In [ ]:
# Load model and tokenizer (from local dataset, no internet needed)
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, local_files_only=True)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_DIR, local_files_only=True)
model.to(device)
model.eval()
print(f"Model loaded: {model.config.model_type}, params: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Load and preprocess test data
test_df = pd.read_csv(f"{DATA_DIR}/test.csv")
test_df["source"] = test_df["transliteration"].apply(clean_transliteration)
print(f"Test examples: {len(test_df)}")

# Generate translations
results = []
for _, row in test_df.iterrows():
    input_text = PREFIX + row["source"]
    inputs = tokenizer(
        input_text, max_length=MAX_SOURCE_LEN, truncation=True, return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=MAX_TARGET_LEN,
            num_beams=5,
            length_penalty=1.0,
            no_repeat_ngram_size=3,
            early_stopping=True,
        )

    translation = tokenizer.decode(outputs[0], skip_special_tokens=True)
    results.append({"id": int(row["id"]), "translation": translation})
    print(f"ID {row['id']}: {translation[:150]}")

# Write submission
sub_df = pd.DataFrame(results)
sub_df.to_csv("submission.csv", index=False)
print(f"\nsubmission.csv written! ({len(sub_df)} rows)")
sub_df